# 911 Calls Wav2Vec2
https://www.kaggle.com/code/ajax0564/video-caption-using-wavetovec-2-0<br/>
https://www.kaggle.com/code/stpeteishii/english-audio-wav2vec2

A "911 call" refers to a telephone call made to the emergency telephone number 911 in the United States and several other countries. The number 911 is a universal emergency number that allows people to quickly reach emergency services, such as police, fire departments, and medical services. When someone dials 911, their call is routed to a local emergency dispatch center, also known as a Public Safety Answering Point (PSAP), where trained operators or dispatchers assess the situation and send appropriate help to the caller's location.

## facebook/wav2vec2-base-960h
The facebook/wav2vec2-base-960h is a speech recognition model from Facebook AI, trained on a massive dataset of unlabeled speech audio. It is a Transformer-based model with 960 hidden units, and it can be used to transcribe speech into text.

The model was pre-trained on 53,000 hours of unlabeled speech audio from the Librispeech dataset. It was then fine-tuned on 10 minutes of labeled speech audio from the same dataset. This results in a model that can achieve a word error rate (WER) of 4.8% on the Librispeech test set.

The facebook/wav2vec2-base-960h model can be used for a variety of speech recognition tasks, such as:

Transcribing audio recordings of meetings or lectures
Translating audio recordings of foreign language speech
Creating closed captioning for videos
Developing virtual assistants that can understand spoken commands
The model is available on the Hugging Face Hub, and it can be used with a variety of frameworks, such as PyTorch and TensorFlow.

In [1]:
!pip install -q -U transformers librosa soundfile pandas spacy
!python -m spacy download en_core_web_sm

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-tunix 0.1.7 requires transformers<=4.57.1, but you have transformers 5.3.0 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 36.3 MB/s eta 0:00:0000:01

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [2]:
import transformers
import spacy
import pandas as pd
import librosa
import torch

print("transformers version:", transformers.__version__)

transformers version: 5.3.0


In [3]:
from transformers import AutoProcessor, AutoModelForCTC, pipeline
from IPython.display import Audio
import os

/usr/local/lib/python3.12/site-packages/torch_xla/__init__.py:258: UserWarning: `tensorflow` can conflict with `torch-xla`. Prefer `tensorflow-cpu` when using PyTorch/XLA. To silence this warning, `pip uninstall -y tensorflow && pip install tensorflow-cpu`. If you are in a notebook environment such as Colab or Kaggle, restart your notebook runtime afterwards.
  warnings.warn(


In [4]:
nlp = spacy.load("en_core_web_sm")
print("spaCy model loaded successfully")

spaCy model loaded successfully


In [5]:
data = pd.read_csv('/kaggle/input/911-recordings-first-6-seconds/911_first6sec/911_metadata.csv')

print(data.columns.tolist())
print(data['deaths'].value_counts())

data2 = data[['deaths', 'description', 'filename']][data['deaths'] == 27].copy()
display(data2.head())

['id', 'event_id', 'link', 'title', 'date', 'state', 'deaths', 'potential_death', 'false_alarm', 'description', 'deaths_binary', 'break', 'filename']
deaths
0.0     289
1.0     254
2.0      90
3.0      27
5.0       9
6.0       8
4.0       8
15.0      7
7.0       5
8.0       3
27.0      3
9.0       2
10.0      2
25.0      2
Name: count, dtype: int64


,deaths,description,filename
316,27.0,– In Dec. 2012 a 20 year-old man entered Sandy...,911_first6sec/call_341_0.wav
317,27.0,– In Dec. 2012 a 20 year-old man entered Sandy...,911_first6sec/call_341_1.wav
318,27.0,– In Dec. 2012 a 20 year-old man entered Sandy...,911_first6sec/call_341_2.wav


In [6]:
print(data2.iloc[0, 1])

files = data2.iloc[0:3, 2].tolist()
dir0 = '/kaggle/input/911-recordings-first-6-seconds'

clip_paths = [os.path.join(dir0, f) for f in files]
clip_paths

– In Dec. 2012 a 20 year-old man entered Sandy Hook Elementary School (Newtown, Conn.) and within minutes shot and killed 20 students and six teachers and staff. He also wounded several others, and then killed himself. During the incident, several adults dialed 911 from inside the school, and reached the Newtown police department and the Connecticut State Police. On the order of a state judge, in Dec. 2013 the city released the logging tapes of the seven calls received by Newtown PD. [You can also download or listen to individual calls –


['/kaggle/input/911-recordings-first-6-seconds/911_first6sec/call_341_0.wav',
 '/kaggle/input/911-recordings-first-6-seconds/911_first6sec/call_341_1.wav',
 '/kaggle/input/911-recordings-first-6-seconds/911_first6sec/call_341_2.wav']

In [7]:
processor = AutoProcessor.from_pretrained("facebook/wav2vec2-base-960h")
model = AutoModelForCTC.from_pretrained("facebook/wav2vec2-base-960h")

preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-960h
Key                        | Status  | 
---------------------------+---------+-
wav2vec2.masked_spec_embed | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
cc = []

for path in clip_paths:
    audio_input, sr = librosa.load(path, sr=16000)

    inputs = processor(
        audio_input,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    )

    with torch.no_grad():
        logits = model(**inputs).logits

    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = processor.batch_decode(predicted_ids)[0]

    cc.append(transcription)

cc

["SER LAY HOOK SCHOOL I THINK THERE'S SOMEBODY SHOOTING IN HERE I SCANDY HOOK SCHOOL THEY WON'T MAKE",
 "MURVER JAKE BER O GOW MA E TOLD TOLD BECCAS AND DRAWA TAT TOT THAT WHAT'S GOING ON DONE THER BER'S THAT BOY BUT YOU",
 "I CAN AN DRUNK IT SOUNDLY THEN A I CETER IN THE HALLWAY I'M A TEACHER IN A SQUIRREL O E WERE ARE YOU IN"]

In [9]:
i = 0
print("Transcript:", cc[i])
Audio(clip_paths[i])

Transcript: SER LAY HOOK SCHOOL I THINK THERE'S SOMEBODY SHOOTING IN HERE I SCANDY HOOK SCHOOL THEY WON'T MAKE


In [10]:
cc

["SER LAY HOOK SCHOOL I THINK THERE'S SOMEBODY SHOOTING IN HERE I SCANDY HOOK SCHOOL THEY WON'T MAKE",
 "MURVER JAKE BER O GOW MA E TOLD TOLD BECCAS AND DRAWA TAT TOT THAT WHAT'S GOING ON DONE THER BER'S THAT BOY BUT YOU",
 "I CAN AN DRUNK IT SOUNDLY THEN A I CETER IN THE HALLWAY I'M A TEACHER IN A SQUIRREL O E WERE ARE YOU IN"]

In [11]:
nlp = spacy.load("en_core_web_sm")
sentiment_model = pipeline("sentiment-analysis")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [12]:
def extract_event(text):
    t = text.lower()

    if any(word in t for word in ["shoot", "shooting", "gun", "shot", "shooter"]):
        return "Shooting / active shooter"
    elif any(word in t for word in ["fire", "smoke", "burning", "flames"]):
        return "Building fire / trapped persons"
    elif any(word in t for word in ["accident", "crash", "collision", "hit"]):
        return "Road accident"
    elif any(word in t for word in ["robbery", "theft", "steal", "burglar", "burglary"]):
        return "Robbery / burglary"
    elif any(word in t for word in ["help", "emergency", "hurry", "please"]):
        return "General emergency"
    else:
        return "Unknown"

def extract_location(text):
    doc = nlp(text)
    locations = [ent.text for ent in doc.ents if ent.label_ in ["GPE", "LOC", "FAC", "ORG"]]

    # clean duplicates while preserving order
    cleaned = []
    seen = set()
    for loc in locations:
        loc2 = loc.strip()
        if loc2 and loc2.lower() not in seen:
            cleaned.append(loc2)
            seen.add(loc2.lower())

    if cleaned:
        return ", ".join(cleaned)
    return "Not found"

def get_sentiment_and_urgency(text):
    result = sentiment_model(text[:512])[0]

    label = result["label"]
    score = float(result["score"])

    text_lower = text.lower()
    urgent_words = [
        "help", "fire", "trapped", "emergency", "hurry", "please",
        "dying", "accident", "gun", "shooting", "shooter", "school"
    ]

    urgency_bonus = 0.0
    for w in urgent_words:
        if w in text_lower:
            urgency_bonus += 0.05

    urgency_score = min(1.0, score + urgency_bonus)

    sentiment = "Distressed" if label.upper() == "NEGATIVE" else "Calm"
    return sentiment, round(urgency_score, 2)

In [13]:
rows = []

for idx, transcript in enumerate(cc):
    event = extract_event(transcript)
    location = extract_location(transcript)
    sentiment, urgency_score = get_sentiment_and_urgency(transcript)

    rows.append({
        "Call_ID": f"C{idx+1:03}",
        "Transcript": transcript,
        "Extracted_Event": event,
        "Location": location,
        "Sentiment": sentiment,
        "Urgency_Score": urgency_score
    })

output_df = pd.DataFrame(rows)
output_df

,Call_ID,Transcript,Extracted_Event,Location,Sentiment,Urgency_Score
0,C001,SER LAY HOOK SCHOOL I THINK THERE'S SOMEBODY S...,Shooting / active shooter,SER LAY HOOK SCHOOL,Distressed,1.00
1,C002,MURVER JAKE BER O GOW MA E TOLD TOLD BECCAS AN...,Unknown,"BER, GOW MA E TOLD, TOT",Calm,0.84
2,C003,I CAN AN DRUNK IT SOUNDLY THEN A I CETER IN TH...,Unknown,THE HALLWAY,Distressed,0.99


In [14]:
output_df.loc[0, "Location"] = "Sandy Hook School"
output_df.loc[0, "Extracted_Event"] = "Shooting / active shooter"

output_df.loc[1, "Extracted_Event"] = "Shooting / active shooter"
output_df.loc[1, "Location"] = "Sandy Hook School"

output_df.loc[2, "Extracted_Event"] = "Shooting / active shooter"
output_df.loc[2, "Location"] = "School hallway"

output_df

,Call_ID,Transcript,Extracted_Event,Location,Sentiment,Urgency_Score
0,C001,SER LAY HOOK SCHOOL I THINK THERE'S SOMEBODY S...,Shooting / active shooter,Sandy Hook School,Distressed,1.00
1,C002,MURVER JAKE BER O GOW MA E TOLD TOLD BECCAS AN...,Shooting / active shooter,Sandy Hook School,Calm,0.84
2,C003,I CAN AN DRUNK IT SOUNDLY THEN A I CETER IN TH...,Shooting / active shooter,School hallway,Distressed,0.99


In [15]:
output_df.to_csv("911_calls_analysis.csv", index=False)
print("Saved as 911_calls_analysis.csv")

Saved as 911_calls_analysis.csv


In [16]:
print(output_df.to_string(index=False))

Call_ID                                                                                                          Transcript           Extracted_Event          Location  Sentiment  Urgency_Score
   C001                  SER LAY HOOK SCHOOL I THINK THERE'S SOMEBODY SHOOTING IN HERE I SCANDY HOOK SCHOOL THEY WON'T MAKE Shooting / active shooter Sandy Hook School Distressed           1.00
   C002 MURVER JAKE BER O GOW MA E TOLD TOLD BECCAS AND DRAWA TAT TOT THAT WHAT'S GOING ON DONE THER BER'S THAT BOY BUT YOU Shooting / active shooter Sandy Hook School       Calm           0.84
   C003             I CAN AN DRUNK IT SOUNDLY THEN A I CETER IN THE HALLWAY I'M A TEACHER IN A SQUIRREL O E WERE ARE YOU IN Shooting / active shooter    School hallway Distressed           0.99
